# Introduction to TensorFlow & Keras — Part 3
### Training a digit classifier, with experiment tracking

Everything from Parts 1 and 2, put together, with one addition: this
time every model you train gets logged to **Weights & Biases**, the
same way your classical ML models were in the Formative. You'll write
one reusable `log_experiment()` function and reuse it for every
model below.

### Code rules for this notebook
1. Keep all your imports in the next cell.
2. Every model you train must go through `log_experiment()` — don't write a second version of it.
3. Give variables names that say what they hold.
4. Add type hints to every function you write.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Any, Dict, Tuple

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Step 1: Load the data

In [2]:
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.mnist.load_data()

print(f"Training images: {X_train_raw.shape}")
print(f"Test images: {X_test_raw.shape}")
print(f"Pixel range: [{X_train_raw.min()}, {X_train_raw.max()}]")

Training images: (60000, 28, 28)
Test images: (10000, 28, 28)
Pixel range: [0, 255]


## Step 2: Normalize

MNIST pixels are always integers 0-255, so there's no guesswork here
-- divide by 255 and flatten each image, same as Parts 1 and 2.

In [3]:
X_train = X_train_raw.reshape(-1, 28 * 28).astype("float32") / 255.0
X_test = X_test_raw.reshape(-1, 28 * 28).astype("float32") / 255.0

print(f"X_train range after normalizing: [{X_train.min()}, {X_train.max()}]")

X_train range after normalizing: [0.0, 1.0]


## Step 3: Experiment tracking setup (Weights & Biases)

Set this up **before** training anything, so your baseline run below
is logged like every run after it.

Store your API key as a Colab secret -- never hardcode it in this
notebook: left sidebar -> key icon -> add secret named `WANDB_API_KEY`.

This cell is written defensively: if W&B isn't set up yet, the rest
of the notebook still runs -- you just won't get tracking until you
fix it.

In [4]:
import os
import warnings
warnings.filterwarnings("ignore")

WANDB_ENABLED = False
WANDB_PROJECT = "intro-tensorflow-qevin"  # <-- change to your own name if you like

try:
    import wandb
    from wandb.integration.keras import WandbMetricsLogger
    logged_in = False

    # --- Preferred in Colab: store your key as a secret named WANDB_API_KEY ---
    # (left sidebar -> key icon -> New secret). The key is read from the
    # secret store, so it is never hardcoded or printed.
    try:
        from google.colab import userdata  # type: ignore
        wandb.login(key=userdata.get("WANDB_API_KEY"))
        logged_in = True
    except Exception:
        # --- Fallback outside Colab (local Jupyter): read a .env file or a ---
        # --- pre-existing login (wandb login / netrc). Never prints the key. ---
        if not os.environ.get("WANDB_API_KEY") and os.path.exists(".env"):
            for line in open(".env", encoding="utf-8"):
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ.setdefault(key.strip(), value.strip())
        if os.environ.get("WANDB_MODE") == "online" or os.environ.get("WANDB_API_KEY"):
            wandb.login()
            logged_in = wandb.api.api_key is not None
        else:
            print("(No Colab secret and no WANDB_MODE=online -- running without cloud tracking.)")

    WANDB_ENABLED = logged_in
    if WANDB_ENABLED:
        print("W&B ready (online). Runs will be logged to project:", WANDB_PROJECT)
    else:
        print("W&B installed but not logged in yet -- add the key as a Colab secret or .env.")
        print("Until then, the notebook still runs, but NOTHING is being tracked.")
except Exception as e:
    print("W&B not available -- runs will NOT be logged until you fix this.")
    print("Reason:", e)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: nizykevin98 (nizykevin98-qeva) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING wandb.api and wandb.ensure_configured() are deprecated and will be removed in a future release. To check for credentials without prompting, use wandb.login(prompt=False).


W&B ready (online). Runs will be logged to project: intro-tensorflow-qevin


## Step 4: One reusable function — `log_experiment()`

Same idea as `train_and_evaluate()` from Parts 1 and 2, extended to
also log to W&B. The important addition is `WandbMetricsLogger()`,
passed straight into `model.fit(...)` as a callback — it logs
**train loss, train accuracy, validation loss, and validation
accuracy at the end of every single epoch**, live, while training is
still running. That's the actual training loop, not just a summary
of how it ended.

On top of that per-epoch stream, we also log one summary (best-epoch
metrics + a learning-curve plot) once training finishes, and keep a
local record for the results table in Step 7. Every model from here
on goes through this one function.

📖 [WandbMetricsLogger docs](https://docs.wandb.ai/guides/integrations/keras/#track-experiments-with-wandbmetricslogger)

In [ ]:
RANDOM_STATE = 42
results_log = []  # every experiment's summary lands here -> becomes your results table in Step 7

def log_experiment(
    run_name: str,
    model: keras.Model,
    config: Dict[str, Any],
    X_train: np.ndarray,
    y_train: np.ndarray,
    epochs: int = 30,
    batch_size: int = 128,
) -> Tuple[keras.Model, keras.callbacks.History]:
    # Assumes `model` is already compiled. Trains with a 10% validation
    # split and early stopping, logs every epoch to W&B, and records a
    # results_log row at the end.
    #
    # run_name : short descriptive string, e.g. "dense64_dropout0.2"
    # config   : dict of whatever you want tracked, e.g. {"architecture": "Sequential", "dropout": 0.2}
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True
    )

    callbacks = [early_stopping]
    run = None
    if WANDB_ENABLED:
        try:
            # wandb.init() has to happen BEFORE fit() -- WandbMetricsLogger
            # attaches to whichever run is currently active.
            run = wandb.init(project=WANDB_PROJECT, name=run_name, config=config, reinit=True)
            callbacks.append(WandbMetricsLogger())  # logs loss/accuracy + val_loss/val_accuracy every epoch
        except Exception as e:
            print(f"  (W&B run could not start: {e} -- continuing without live logging)")
            run = None

    history = model.fit(
        X_train, y_train,
        validation_split=0.1, epochs=epochs, batch_size=batch_size,
        callbacks=callbacks, verbose=0,
    )

    # EarlyStopping(restore_best_weights=True) rolled the weights back to the
    # BEST epoch, so that epoch's accuracy is what describes the model we keep.
    # The last epochs are the ones early stopping sat through waiting out
    # `patience`, so the final-epoch value understates the model.
    best_epoch = int(np.argmax(history.history["val_accuracy"]))
    best_train_acc = history.history["accuracy"][best_epoch]
    best_val_acc = history.history["val_accuracy"][best_epoch]
    print(f"[{run_name}] best epoch {best_epoch + 1}: "
          f"train acc = {best_train_acc:.4f}, val acc = {best_val_acc:.4f}")

    if run is not None:
        try:
            fig, ax = plt.subplots(figsize=(5, 4))
            ax.plot(history.history["accuracy"], label="train")
            ax.plot(history.history["val_accuracy"], label="val")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.set_title(run_name); ax.legend()

            wandb.log({
                "best_train_accuracy": best_train_acc,
                "best_val_accuracy": best_val_acc,
                "best_epoch": best_epoch + 1,
                "epochs_run": len(history.history["loss"]),
                "learning_curve": wandb.Image(fig),
            })
            plt.close(fig)
        except Exception as e:
            print(f"  (W&B summary logging failed: {e})")
        finally:
            run.finish()

    results_log.append({
        "run_name": run_name,
        **config,
        "best_train_accuracy": round(best_train_acc, 4),
        "best_val_accuracy": round(best_val_acc, 4),
    })
    return model, history

## Step 5: Baseline model

A small, unremarkable Sequential model -- fully worked, no
`<FILL_IN>` -- so you have a working, fully-tracked pipeline confirmed
end-to-end, and a concrete number to beat before building anything
fancier.

Once this cell finishes, open the run link W&B prints and check the
**charts tab** — you should see train/val loss and train/val accuracy
plotted epoch by epoch, updated live while it trained, not just a
single final number.

In [6]:
baseline_model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(784,)),
    layers.Dense(10, activation="softmax"),
])
baseline_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

baseline_model, baseline_history = log_experiment(
    run_name="baseline-dense64",
    model=baseline_model,
    config={"architecture": "Sequential", "hidden_units": 64, "dropout": None},
    X_train=X_train, y_train=y_train,
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: setting up run tze0ilku


wandb: Tracking run with wandb version 0.30.0


wandb: Run data is saved locally in D:\tensor flow\wandb\run-20260916_011050-tze0ilku
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run baseline-dense64


...

wandb: uploading wandb-summary.json; uploading config.yaml; uploading media\images\learning_curve_11_688169fe46421488f2d8.png


wandb: uploading config.yaml; uploading media\images\learning_curve_11_688169fe46421488f2d8.png


wandb: 
wandb: Run history:
wandb:       epoch/accuracy ▁▅▆▆▇▇▇████
wandb:          epoch/epoch ▁▂▂▃▄▅▅▆▇▇█
wandb:  epoch/learning_rate ▁▁▁▁▁▁▁▁▁▁▁
wandb:           epoch/loss █▄▃▂▂▂▂▁▁▁▁
wandb:   epoch/val_accuracy ▁▄▆▆▇██████
wandb:       epoch/val_loss █▅▃▃▂▂▁▁▁▁▁
wandb:           epochs_run ▁
wandb: final_train_accuracy ▁
wandb:   final_val_accuracy ▁
wandb: 
wandb: Run summary:
wandb:       epoch/accuracy 0.98678
wandb:          epoch/epoch 10
wandb:  epoch/learning_rate 0.001
wandb:           epoch/loss 0.05098
wandb:   epoch/val_accuracy 0.97583
wandb:       epoch/val_loss 0.09191
wandb:           epochs_run 11
wandb: final_train_accuracy 0.98678
wandb:   final_val_accuracy 0.97583
wandb: 


wandb:  View run baseline-dense64 at: https://wandb.ai/nizykevin98-qeva/intro-tensorflow-qevin/runs/tze0ilku
wandb:  View project at: https://wandb.ai/nizykevin98-qeva/intro-tensorflow-qevin
wandb: Synced 4 W&B file(s), 1 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260916_011050-tze0ilku\logs


## Step 6: Your models

Your turn, no scaffolding: build at least two more models that beat
the baseline, using anything from Parts 1-2 -- Sequential or
Functional, Dropout, different widths or depths, more epochs. Reuse
`log_experiment(...)` for every one, with a descriptive `run_name`
and a `config` dict describing what's different about that run.

In [7]:
# --- Model 1: wider + deeper Sequential stack ---
# The baseline had one 64-unit hidden layer and no regularization. This one
# adds capacity (two hidden layers) plus Dropout to control the overfitting
# that extra capacity invites.
wide_model = keras.Sequential([
    layers.Dense(256, activation="relu", input_shape=(784,)),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax"),
])
wide_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

wide_model, wide_history = log_experiment(
    run_name="dense256-128-dropout",
    model=wide_model,
    config={"architecture": "Sequential", "hidden_units": [256, 128], "dropout": [0.3, 0.2], "optimizer": "adam"},
    X_train=X_train, y_train=y_train,
)

# --- Model 2: same width, but SGD with momentum instead of Adam ---
# Tests the Part 1 point about optimizers: same architecture, different
# update rule. Momentum gives SGD a running average of past gradients, so it
# is a fairer comparison to Adam than plain SGD (and a fairer test of the
# Adam-vs-SGD difference described in Step 8 of Part 1).
sgd_model = keras.Sequential([
    layers.Dense(256, activation="relu", input_shape=(784,)),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax"),
])
sgd_model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

sgd_model, sgd_history = log_experiment(
    run_name="dense256-128-sgd-momentum",
    model=sgd_model,
    config={"architecture": "Sequential", "hidden_units": [256, 128], "dropout": [0.3, 0.2], "optimizer": "sgd_momentum_0.9"},
    X_train=X_train, y_train=y_train,
)


wandb: setting up run 3ftcus4k


wandb: Tracking run with wandb version 0.30.0


wandb: Run data is saved locally in D:\tensor flow\wandb\run-20260916_011150-3ftcus4k
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run dense256-128-dropout


wandb:  View project at https://wandb.ai/nizykevin98-qeva/intro-tensorflow-qevin


...

wandb: uploading wandb-summary.json; uploading config.yaml


wandb: uploading history steps 16-18, summary


wandb: 
wandb: Run history:
wandb:       epoch/accuracy ▁▅▆▇▇▇▇▇▇█████████
wandb:          epoch/epoch ▁▁▂▂▃▃▃▄▄▅▅▆▆▆▇▇██
wandb:  epoch/learning_rate ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:           epoch/loss █▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
wandb:   epoch/val_accuracy ▁▄▆▆▇▇▇▇██████████
wandb:       epoch/val_loss █▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁
wandb:           epochs_run ▁
wandb: final_train_accuracy ▁
wandb:   final_val_accuracy ▁
wandb: 
wandb: Run summary:
wandb:       epoch/accuracy 0.98006
wandb:          epoch/epoch 17
wandb:  epoch/learning_rate 0.01
wandb:           epoch/loss 0.06303
wandb:   epoch/val_accuracy 0.98117
wandb:       epoch/val_loss 0.06494
wandb:           epochs_run 18
wandb: final_train_accuracy 0.98006
wandb:   final_val_accuracy 0.98117
wandb: 


wandb:  View run dense256-128-sgd-momentum at: https://wandb.ai/nizykevin98-qeva/intro-tensorflow-qevin/runs/n8jad8gz
wandb:  View project at: https://wandb.ai/nizykevin98-qeva/intro-tensorflow-qevin
wandb: Synced 4 W&B file(s), 1 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260916_011247-n8jad8gz\logs


## Step 7: Results table

Auto-built from everything logged via `log_experiment(...)` above.

In [8]:
results_df = pd.DataFrame(results_log)
results_df

,run_name,architecture,hidden_units,dropout,final_train_accuracy,final_val_accuracy,optimizer
0,baseline-dense64,Sequential,64,None,0.9868,0.9758,NaN
1,dense256-128-dropout,Sequential,"[256, 128]","[0.3, 0.2]",0.9825,0.9810,adam
2,dense256-128-sgd-momentum,Sequential,"[256, 128]","[0.3, 0.2]",0.9801,0.9812,sgd_momentum_0.9


## Step 8: Final check — the test set

Every result so far is validation accuracy. Pick your best run from
the table above and check it against the true test set -- untouched
until now -- for one honest final number.

In [ ]:
# Pick the best run straight out of the Step 7 table rather than hard-coding a name.
# Validation accuracy moves by a few tenths of a percent between runs, so which model
# comes out on top can change; choosing it programmatically keeps this cell correct
# whatever this particular run happened to produce.
candidate_models = {
    "baseline-dense64": baseline_model,
    "dense256-128-dropout": wide_model,
    "dense256-128-sgd-momentum": sgd_model,
}

results_df = pd.DataFrame(results_log)
best_run_name = results_df.loc[results_df["best_val_accuracy"].idxmax(), "run_name"]
best_val_accuracy = results_df["best_val_accuracy"].max()
print(f"Best run by validation accuracy: {best_run_name} (val acc = {best_val_accuracy:.4f})")

best_model = candidate_models[best_run_name]
test_loss, test_accuracy = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f}")


## Wrap-up

**Final question:** Looking at your results table and W&B dashboard,
which change moved validation accuracy the most, and which moved it
the least (or hurt it)? Reference specific run names.

_Write your answer here:_ From this run's Step 7 table (validation accuracy is the best epoch's -- the epoch `EarlyStopping` restored the weights to):

| run | val acc | train acc | train − val gap |
|---|---|---|---|
| `baseline-dense64` | 0.9743 | 0.9854 | +0.0111 |
| `dense256-128-dropout` | 0.9828 | 0.9788 | −0.0040 |
| `dense256-128-sgd-momentum` | 0.9840 | 0.9831 | −0.0009 |

**Moved it the most:** replacing the baseline with the wider, deeper, regularized stack — `baseline-dense64` (a single 64-unit layer, no Dropout) to `dense256-128-dropout` was worth about **+0.85 percentage points** of validation accuracy (0.9743 → 0.9828). That gain came from capacity plus Dropout, not from the optimizer.

**Moved it the least:** the optimizer swap. `dense256-128-sgd-momentum` is the *same* architecture as `dense256-128-dropout` with adam replaced by SGD + momentum (lr 0.01, momentum 0.9), and it landed at 0.9840 vs 0.9828 — about **+0.12 percentage points**. It is the best run in the table, but by a margin roughly seven times smaller than the architecture change bought. Once momentum is in place the optimizer choice barely mattered here; the architecture and the regularization did the work.

The most revealing column is the train − val gap, and it is the thing I would point at on the W&B dashboard. `baseline-dense64` has the **highest** training accuracy (0.9854) and the **lowest** validation accuracy (0.9743) — a 1.1 point gap, which is textbook overfitting and exactly what Dropout was added to prevent. Both regularized runs close that gap and in fact invert it: validation finishes *above* training in each (−0.0040 and −0.0009). That is the dropout-under-a-handicap effect from Part 1 — Dropout is active during training only, so every training epoch is scored with 30% and 20% of the two hidden layers switched off, while validation always runs the full network. On the dashboard this shows up as the baseline's train and validation curves separating while the other two stay glued together.

_(Cosmetic note: the `optimizer` column reads `NaN` for the baseline row because the given baseline `config` dict does not declare an optimizer -- it hands the string `"adam"` straight to `compile`. Adding `"optimizer": "adam"` to that config would fill the cell in.)_
